# 🏠 House Price Predictor — Exploratory Data Analysis

**Dataset**: Ames Housing (from Kaggle)

This notebook covers:
1. Data overview & shape
2. Missing value analysis
3. Target variable distribution
4. Correlation heatmap
5. Key feature relationships

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '../src')
from data_processing import load_data, summarize_missing, NUMERIC_FEATURES

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Blues_r')

df = load_data('../data/train.csv')
df.head()

## 1. Dataset Overview

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:")
print(df.dtypes.value_counts())
df.describe()

## 2. Missing Values

In [ ]:
missing = summarize_missing(df)
print(f"Columns with missing data: {len(missing)}\n")

fig, ax = plt.subplots(figsize=(10, 6))
missing.head(20)['missing_pct'].plot(kind='barh', ax=ax, color='#c0392b', alpha=0.85)
ax.set_xlabel('Missing (%)')
ax.set_title('Top 20 Columns by Missing Data', fontweight='bold')
plt.tight_layout()
plt.show()

missing.head(20)

## 3. Target Variable — SalePrice

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
axes[0].hist(df['SalePrice'], bins=50, color='#2563eb', alpha=0.8, edgecolor='white')
axes[0].set_title('SalePrice Distribution (raw)', fontweight='bold')
axes[0].set_xlabel('Sale Price ($)')

# Log-transformed
axes[1].hist(np.log1p(df['SalePrice']), bins=50, color='#1a1612', alpha=0.8, edgecolor='white')
axes[1].set_title('SalePrice Distribution (log)', fontweight='bold')
axes[1].set_xlabel('log(Sale Price)')

plt.suptitle('Target Variable Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(df['SalePrice'].describe())
print(f"\nSkewness: {df['SalePrice'].skew():.3f}")

## 4. Correlation Heatmap

In [ ]:
cols = [c for c in NUMERIC_FEATURES if c in df.columns] + ['SalePrice']
corr = df[cols].corr()

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 7})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Key Feature Relationships

In [ ]:
top_corr = corr['SalePrice'].drop('SalePrice').abs().sort_values(ascending=False).head(6).index.tolist()
print('Top correlated features:', top_corr)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, feat in zip(axes.flat, top_corr):
    ax.scatter(df[feat], df['SalePrice'], alpha=0.3, s=15, color='#2563eb')
    ax.set_xlabel(feat)
    ax.set_ylabel('SalePrice')
    ax.set_title(f'{feat} vs SalePrice', fontweight='bold')

plt.suptitle('Top Features vs Sale Price', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Neighbourhood Price Distribution

In [ ]:
order = df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False).index

plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x='Neighborhood', y='SalePrice', order=order,
            color='#2563eb', fliersize=2)
plt.xticks(rotation=45, ha='right')
plt.title('Sale Price by Neighbourhood', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Key Takeaways

- **SalePrice is right-skewed** — a log transform may help linear models.
- **OverallQual** has the highest correlation with price (~0.79).
- **GrLivArea** and **TotalBsmtSF** are strong numeric predictors.
- Several features have **>50% missing** (e.g. PoolQC, Alley) — we drop or impute them.
- **Neighbourhood** is a strong categorical predictor with wide price spread.

➡️ See `02_modeling.ipynb` for model training and evaluation.